In [ ]:
from qdrant_client import QdrantClient
from qdrant_client.models import VectorParams, Distance, PointStruct

import pandas as pd
import openai
from typing import Any, Hashable

### Read the sampled dataset with Amazon inventory data


In [ ]:
df_items = pd.read_json("../../data/meta_Electronics_2022_2023_with_category_ratings_100_sample_1000.jsonl", lines=True)

In [ ]:
df_items.head()

In [ ]:
list[tuple[Hashable, Any]](df_items["images"].items())[0]

### Preprocess title and features

In [ ]:
def preprocess_description(row):
    return f"{row['title']} {' '.join(row['features'])} {row['description']}"

In [ ]:
def extract_first_large_image(row) : 
    return row['images'][0].get("large","")

In [ ]:
df_items["description"] = df_items.apply(preprocess_description, axis=1)
df_items["image"] = df_items.apply(extract_first_large_image, axis=1)


In [ ]:
df_items.head()

In [ ]:
list[tuple[Hashable, Any]](df_items["description"].items())[0]

### Sample 50 items from the dataset

In [ ]:
df_sample = df_items.sample(n=50, random_state=42)

In [ ]:
len(df_sample)

In [ ]:
data_to_embed = df_sample[["description", "image","rating_number","price","average_rating","parent_asin"]].to_dict(orient="records")

In [ ]:
data_to_embed

### Define the embedding function

In [ ]:
response = openai.embeddings.create(
    input="Randome text",
    model="text-embedding-3-small"
)

In [ ]:
len(response.data[0].embedding)

In [ ]:
def get_embedding(text , model="text-embedding-3-small"):
    response = openai.embeddings.create(
        input = text,
        model = model,
    )
    return response.data[0].embedding

In [ ]:
get_embedding("Hi")

### Create Qdrant Collection

In [27]:
qdrant_client = QdrantClient(url="http://localhost:6333")

In [28]:
qdrant_client.create_collection(
    collection_name="Amazon-items-collection-00",
    vectors_config=VectorParams(size=1536, distance=Distance.COSINE)
)

True

### Embed data

### Test

In [29]:
pointstruct = PointStruct(
    id=0,
    vector=get_embedding("Test text"),
    payload={
        "text":"Test text",
        "model":"text-embedding-3-small"
    }
)

In [30]:
pointstruct

PointStruct(id=0, vector=[-0.020111083984375, 0.0070037841796875, 0.0377197265625, -0.040252685546875, -0.0191802978515625, -0.0343017578125, 0.000576019287109375, -0.024261474609375, 0.038665771484375, 0.0015888214111328125, 0.0306243896484375, 0.01277923583984375, -0.01087188720703125, 0.011260986328125, 0.026397705078125, 0.04302978515625, -0.04681396484375, -0.006893157958984375, -0.021728515625, 0.05108642578125, 0.007061004638671875, 0.01308441162109375, 0.01129150390625, -0.032501220703125, -0.003940582275390625, -0.0390625, -0.03692626953125, 0.0050048828125, 0.058441162109375, -0.07452392578125, 0.0313720703125, -0.0440673828125, -0.0002703666687011719, -0.010955810546875, 0.00604248046875, 0.0382080078125, 0.0303192138671875, 0.03759765625, 0.0102386474609375, -0.02935791015625, -0.014923095703125, -0.01042938232421875, 0.021270751953125, 0.0181427001953125, 0.02667236328125, 0.00891876220703125, -0.01177978515625, 0.00879669189453125, 0.01995849609375, 0.037933349609375, -0.

### Amazon data

In [31]:
pointstruct=[]
for i , data in enumerate[dict](data_to_embed):
    embedding = get_embedding(data["description"])
    pointstruct.append(
        PointStruct(
            id=i,
            vector=embedding,
            payload=data
        )
    )

In [32]:
pointstruct

[PointStruct(id=0, vector=[-0.0165557861328125, -0.01218414306640625, -0.0022220611572265625, 0.0316162109375, -0.032958984375, -0.0955810546875, 0.0261383056640625, 0.0316162109375, 0.019989013671875, 0.016632080078125, 0.0219879150390625, -0.00104522705078125, -0.059326171875, 0.004276275634765625, 0.01200103759765625, -0.0180206298828125, -0.05780029296875, 0.0338134765625, -0.007274627685546875, 0.00202178955078125, 0.019256591796875, 0.00966644287109375, 0.0150146484375, -0.0014019012451171875, 0.015594482421875, -0.041900634765625, 0.023773193359375, -0.02117919921875, 0.01239013671875, -0.0215911865234375, -0.046844482421875, -0.036865234375, 0.002666473388671875, -0.008148193359375, -0.056182861328125, 0.00677490234375, 7.092952728271484e-06, 0.0026035308837890625, -0.01611328125, -0.015167236328125, 0.0303802490234375, 0.0203857421875, 0.05303955078125, 0.015869140625, -0.0272064208984375, -0.03155517578125, -0.053619384765625, 0.040771484375, -0.0033473968505859375, -0.021865

In [33]:
len(pointstruct)

50

### Write embedded data to Qdrant

In [37]:
qdrant_client.upsert(
    collection_name="Amazon-items-collection-00",
    wait=True,
    points=pointstruct,
)

UpdateResult(operation_id=1, status=<UpdateStatus.COMPLETED: 'completed'>)

### Define a function for Data retrieval

In [38]:
def retrieve_data(query, k=5):
    query_embedding = get_embedding(query)
    results = qdrant_client.query_points(
        collection_name="Amazon-items-collection-00",
        query=query_embedding,
        limit=k
    )
    return results

### Test retrieval

In [39]:
retrieve_data("what kind of charging cords do you offer?", k=10).points

[ScoredPoint(id=46, version=1, score=0.49119475, payload={'description': 'NetDot Magnetic Charging Cable, Gen10 Nylon Braided Magnetic Phone Charger Compatible with USB-C and Micro USB Devices (5ft/3 Pack Black),10G3IN11.5mblack3 New technology: latest version of magnetic cables, better compatibility and stability. Fast Charging: Support 9V/2a fast charging & sync for Android smart phones. Convenience: Easy to connect or disconnect, protect charging port, convenient to charge in the dark, connectors can be used as Anti-dust plug. Caution: not interchangeable with Gen2/Gen3/Gen5/Gen7 cables or connectors. Service: One year warranty and friendly customer service. []', 'image': 'https://m.media-amazon.com/images/I/412ZIIzNNML._AC_.jpg', 'rating_number': 247, 'price': None, 'average_rating': 4.1, 'parent_asin': 'B07JZ3NB7G'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=38, version=1, score=0.375438, payload={'description': 'PWR+ 65W Laptop Charger for HP 741727-001 7104